<a href="https://colab.research.google.com/github/heyitsmialee/PRAGma/blob/main/pragma_260430.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 양산 DES 속도 예측 모델

In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.7 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import optuna
import joblib
import json
import shap
import matplotlib.pyplot as plt
import os

def load_and_preprocess_data(csv_path='./Train_0319.csv'):
    raw_df = pd.read_csv(csv_path, encoding='cp949')

    df = raw_df[[
        'DRY FILM 정보', '선폭 OFFSET', '양산DES 속도',
        '거래처', '제품군', '공법구분', 'LAYER', '도금구분', '노광 설비정보',
        'DES 설비정보', '정면 설비정보', 'Cu 표면두께 Max_Val', 'Cu 표면두께 AVG_VAL',
        'Cu 표면두께 Min_Val', 'Cu 표면두께 Std_Val', 'Cu 표면두께 Median_Val',
        '재작업사유', '분석치_Etch factor', '분석치_Etching(염화동) - Cu',
        '분석치_Etching(염화동) - HCl', '분석치_Etching(염화동) - 비중',
        '분석치_Etching(염화동) - 온도', '분석치_Etching-첨가제(HB-120EF)',
        '분석치_Etching량', '분석치_Soft Etch - Cu', '분석치_Soft Etch - H2SO4',
        '분석치_Soft Etch - SPS', '분석치_박리액 - 농도', '분석치_수세수 - pH',
        '분석치_현상액 - pH', '분석치_현상액 - 농도'
    ]].copy()

    new_cols = [
        'df_type', 'width_offset', 'mass_des_speed',
        'customer', 'product_family', 'process_type', 'layer_count',
        'plating_type', 'expo_eq_id', 'des_eq_id', 'brush_eq_id', 'cu_thick_max',
        'cu_thick_avg', 'cu_thick_min', 'cu_thick_std', 'cu_thick_median',
        'rework_history', 'etch_factor', 'meas_etch_cu', 'meas_etch_hcl',
        'meas_etch_sg', 'meas_etch_temp', 'meas_etch_additive', 'meas_etch_amount',
        'meas_softetch_cu', 'meas_softetch_h2so4', 'meas_softetch_sps',
        'meas_strip_conc', 'meas_rinse_ph', 'meas_dev_ph', 'meas_dev_conc'
    ]
    df.columns = new_cols

    # 1. 범주형 변수 중 일부는 남기고 불필요한 것 제거
    drop_cols = [
        'df_type', 'customer', 'product_family', 'process_type',
        'layer_count', 'plating_type'
    ]
    df = df.drop(columns=drop_cols)

    # 타겟 변수 분리
    y = df['mass_des_speed']
    x = df.drop(columns=['mass_des_speed'])

    # 범주형 / 수치형 피처 분리
    categorical_features = ['rework_history', 'expo_eq_id', 'des_eq_id', 'brush_eq_id', 'width_offset']
    numeric_features = [col for col in x.columns if col not in categorical_features]

    # 결측치 처리
    x[numeric_features] = x[numeric_features].fillna(x[numeric_features].median())
    for cat_col in categorical_features:
        x[cat_col] = x[cat_col].fillna('Unknown').astype(str)

    # 3. Train_Val / Test 분리
    x_train_val, x_test_set, y_train_val, y_test_set = train_test_split(x, y, test_size=0.2, random_state=42)
    x_train_set, x_val_set, y_train_set, y_val_set = train_test_split(x_train_val, y_train_val, test_size=0.25, random_state=42)

    # 4. 전처리 파이프라인
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ])

    x_train_transformed = preprocessor.fit_transform(x_train_set)
    x_val_transformed = preprocessor.transform(x_val_set)
    x_test_transformed = preprocessor.transform(x_test_set)

    cat_encoder = preprocessor.named_transformers_['cat']
    encoded_cat_cols = cat_encoder.get_feature_names_out(categorical_features)
    final_columns = numeric_features + list(encoded_cat_cols)

    df_train_set = pd.DataFrame(x_train_transformed, columns=final_columns, index=x_train_set.index)
    df_train_set['mass_des_speed'] = y_train_set

    df_val_set = pd.DataFrame(x_val_transformed, columns=final_columns, index=x_val_set.index)
    df_val_set['mass_des_speed'] = y_val_set

    df_test_set = pd.DataFrame(x_test_transformed, columns=final_columns, index=x_test_set.index)
    df_test_set['mass_des_speed'] = y_test_set

    joblib.dump(preprocessor, 'preprocessor.pkl')

    return df_train_set, df_val_set, df_test_set


def select_best_regressor(x_train_data, y_train_data, x_val_data, y_val_data):
    print("--- [1단계] 회귀 모델 선택 (기본 파라미터) ---")
    models = {
        "RandomForest": RandomForestRegressor(random_state=42),
        "GradientBoosting": GradientBoostingRegressor(random_state=42),
        "XGBoost": XGBRegressor(random_state=42),
        "LightGBM": LGBMRegressor(random_state=42, verbose=-1),
        "Ridge": Ridge(random_state=42),
        "ElasticNet": ElasticNet(random_state=42),
        "SVR": SVR()
    }

    best_model_name = None
    best_score = -float('inf')

    for name, model in models.items():
        model.fit(x_train_data, y_train_data)
        pred = model.predict(x_val_data)
        score = r2_score(y_val_data, pred)
        print(f"  {name} Validation R2: {score:.4f}")

        if score > best_score:
            best_score = score
            best_model_name = name

    print(f"-> 선택된 최고 성능 회귀 모델: {best_model_name} (R2: {best_score:.4f})\n")
    return best_model_name


def tune_regressor_hyperparameters(model_name, x_train_data, y_train_data, x_val_data, y_val_data):
    print(f"--- [2단계] {model_name} 하이퍼파라미터 튜닝 ---")
    def objective(trial):
        model = None
        if model_name == "RandomForest":
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 5, 30),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'random_state': 42
            }
            model = RandomForestRegressor(**param)
        elif model_name == "GradientBoosting":
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'random_state': 42
            }
            model = GradientBoostingRegressor(**param)
        elif model_name == "XGBoost":
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'random_state': 42
            }
            model = XGBRegressor(**param)
        elif model_name == "LightGBM":
            param = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'random_state': 42,
                'verbose': -1
            }
            model = LGBMRegressor(**param)
        elif model_name == "Ridge":
            param = {
                'alpha': trial.suggest_float('alpha', 1e-4, 1e2, log=True),
                'random_state': 42
            }
            model = Ridge(**param)
        elif model_name == "ElasticNet":
            param = {
                'alpha': trial.suggest_float('alpha', 1e-4, 1e2, log=True),
                'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
                'random_state': 42
            }
            model = ElasticNet(**param)
        elif model_name == "SVR":
            param = {
                'C': trial.suggest_float('C', 1e-2, 1e2, log=True),
                'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
                'epsilon': trial.suggest_float('epsilon', 1e-3, 1.0, log=True)
            }
            model = SVR(**param)

        model.fit(x_train_data, y_train_data)
        pred = model.predict(x_val_data)
        return r2_score(y_val_data, pred)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10) # 튜닝 횟수 조정 (시간 단축을 위해)

    print(f"최적 파라미터:", study.best_params)

    best_model = None
    if model_name == "RandomForest":
        best_model = RandomForestRegressor(**study.best_params, random_state=42)
    elif model_name == "GradientBoosting":
        best_model = GradientBoostingRegressor(**study.best_params, random_state=42)
    elif model_name == "XGBoost":
        best_model = XGBRegressor(**study.best_params, random_state=42)
    elif model_name == "LightGBM":
        best_model = LGBMRegressor(**study.best_params, random_state=42, verbose=-1)
    elif model_name == "Ridge":
        best_model = Ridge(**study.best_params, random_state=42)
    elif model_name == "ElasticNet":
        best_model = ElasticNet(**study.best_params, random_state=42)
    elif model_name == "SVR":
        best_model = SVR(**study.best_params)

    best_model.fit(x_train_data, y_train_data)
    return best_model

def tune_and_train_regressor(x_train_data, y_train_data, x_val_data, y_val_data):
    best_model_name = select_best_regressor(x_train_data, y_train_data, x_val_data, y_val_data)
    best_model = tune_regressor_hyperparameters(best_model_name, x_train_data, y_train_data, x_val_data, y_val_data)
    return best_model, best_model_name

def train_and_evaluate(df_train_eval, df_val_eval, df_test_eval):
    y_train_eval = df_train_eval['mass_des_speed']
    x_train_eval = df_train_eval.drop(columns=['mass_des_speed'])

    y_val_eval = df_val_eval['mass_des_speed']
    x_val_eval = df_val_eval.drop(columns=['mass_des_speed'])

    y_test_eval = df_test_eval['mass_des_speed']
    x_test_eval = df_test_eval.drop(columns=['mass_des_speed'])

    model, name = tune_and_train_regressor(x_train_eval, y_train_eval, x_val_eval, y_val_eval)

    print("\n=== 데이터셋별 양산 DES 속도 회귀 성능 ===")
    pred = model.predict(x_test_eval)
    sim_r2 = r2_score(y_test_eval, pred)
    print(f"[Test Set] R2 Score: {sim_r2:.4f}")

    joblib.dump(model, f'best_{name}_mass_speed_regressor.pkl')

    with open('feature_columns_regressor.json', 'w', encoding='utf-8') as f:
        json.dump(x_train_eval.columns.tolist(), f, ensure_ascii=False, indent=4)

    return model, name, x_train_eval, x_test_eval, y_test_eval


def generate_shap_analysis(model, x_train_shap, x_test_shap, y_test_shap):
    print("\n--- [4] SHAP 분석 시작 ---")

    if isinstance(model, (RandomForestRegressor, GradientBoostingRegressor, XGBRegressor, LGBMRegressor)):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(x_test_shap)
        expected_value = explainer.expected_value
    else:
        # 모델이 TreeExplainer를 지원하지 않을 경우 KernelExplainer 사용 (매우 느림)
        x_train_summary = shap.kmeans(x_train_shap, 100)
        explainer = shap.KernelExplainer(model.predict, x_train_summary)
        shap_values = explainer.shap_values(x_test_shap)
        expected_value = explainer.expected_value

    if isinstance(expected_value, (list, np.ndarray)):
        expected_value = expected_value[0]

    # 1. SHAP 플롯 이미지 저장
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, x_test_shap, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig("shap_summary_bar.png")
    plt.close()

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, x_test_shap, show=False)
    plt.tight_layout()
    plt.savefig("shap_summary_dot.png")
    plt.close()

    print("SHAP 플롯 이미지 저장 완료 (shap_summary_bar.png, shap_summary_dot.png)")

    # 2. RAG용 마크다운 문서 생성
    generate_rag_text(shap_values, x_test_shap, expected_value, y_test_shap)


def generate_rag_text(shap_values, x_test_rag, expected_value, y_test_rag):
    print("\n--- [5] RAG용 SHAP 분석 마크다운 생성 ---")

    vals = np.abs(shap_values).mean(0)
    feature_names = x_test_rag.columns
    fi_df = pd.DataFrame(list(zip(feature_names, vals)), columns=['feature', 'importance'])
    fi_df.sort_values(by=['importance'], ascending=False, inplace=True)

    # 마크다운 텍스트 구성 시작
    md_content = "# SHAP(SHapley Additive exPlanations) 분석 기반 모델 해석 리포트\n\n"
    md_content += "> 본 리포트는 RAG(Retrieval-Augmented Generation) 시스템이 사용자의 질문에 답할 때 모델의 예측 근거를 제공하기 위해 생성되었습니다.\n\n"

    md_content += "## 1. 글로벌 피처 중요도 (Global Feature Importance)\n"
    md_content += "전체 테스트 데이터를 기준으로, 모델이 **양산 DES 속도**를 예측할 때 가장 크게 의존하는 상위 10개 피처(변수)입니다. 중요도 수치가 높을수록 모델 예측에 미치는 영향력이 큽니다.\n\n"

    md_content += "| 순위 | 변수명 | 평균 SHAP 중요도 (Absolute) |\n"
    md_content += "|---|---|---|\n"
    for i, row in enumerate(fi_df.head(10).itertuples(), 1):
         md_content += f"| {i} | `{row.feature}` | {row.importance:.4f} |\n"
    md_content += "\n"

    md_content += "## 2. 주요 변수별 영향 방향성 (Feature Impact Direction)\n"
    md_content += "상위 5개 주요 변수가 양산 DES 속도의 **증가** 또는 **감소**에 어떻게 기여하는지 분석한 결과입니다.\n\n"

    for feature in fi_df['feature'].head(5):
        feature_idx = list(x_test_rag.columns).index(feature)
        f_shap = shap_values[:, feature_idx]
        f_val = x_test_rag[feature].values

        # 피처 값과 SHAP 값의 상관관계 계산 (Nan 제외)
        mask = ~np.isnan(f_val) & ~np.isnan(f_shap)
        if np.sum(mask) > 1: # 데이터가 충분할 때
            corr = np.corrcoef(f_val[mask], f_shap[mask])[0, 1]
        else:
            corr = 0

        md_content += f"### 변수: `{feature}`\n"
        if corr > 0.3:
             md_content += f"- **경향성**: 양의 상관관계 (이 변수 값이 커질수록 예측 속도를 **높이는** 경향이 있습니다.)\n"
        elif corr < -0.3:
             md_content += f"- **경향성**: 음의 상관관계 (이 변수 값이 커질수록 예측 속도를 **낮추는** 경향이 있습니다.)\n"
        else:
             md_content += f"- **경향성**: 비선형적 또는 특정 구간에서 영향을 미칩니다 (값의 크기만으로 단순 증가/감소를 정의하기 어렵습니다.)\n"
        md_content += "- **설명**: SHAP 플롯(`shap_summary_dot.png`)을 참고하면 이 변수의 세부적인 데이터 분포와 예측치 변동 폭을 확인할 수 있습니다.\n\n"


    md_content += "## 3. 특정 샘플(Local) 예측 해석 예시\n"
    md_content += "모델이 개별 데이터(테스트 셋의 첫 번째 행)를 예측할 때, 어떤 변수들이 어떻게 더하고 빼져서 최종 결과가 나왔는지 보여주는 예시입니다.\n\n"

    md_content += f"- **모델의 평균 기대 예측값(Base Value)**: `{expected_value:.4f}`\n"
    md_content += f"- **실제 타겟(양산 DES 속도) 값**: `{y_test_rag.iloc[0]:.4f}`\n\n"

    # 첫 번째 샘플 분석
    sample_shap = shap_values[0]
    sample_vals = x_test_rag.iloc[0]

    sample_df = pd.DataFrame({
        'Feature': x_test_rag.columns,
        'Feature Value': sample_vals,
        'SHAP Value': sample_shap
    })
    # 영향력이 큰(절대값이 큰) 순서대로 정렬
    sample_df['abs_shap'] = sample_df['SHAP Value'].abs()
    sample_df = sample_df.sort_values(by='abs_shap', ascending=False).head(7)

    prediction = expected_value + np.sum(sample_shap)

    md_content += "| 변수명 | 현재 데이터의 피처 값 | 예측치 변동량 (SHAP) | 변동 방향 |\n"
    md_content += "|---|---|---|---|\n"
    for _, row in sample_df.iterrows():
        direction = "🟢 증가 (Speed Up)" if row['SHAP Value'] > 0 else "🔴 감소 (Speed Down)"
        md_content += f"| `{row['Feature']}` | {row['Feature Value']:.4f} | {row['SHAP Value']:.4f} | {direction} |\n"

    md_content += f"\n- 위와 같이 각 변수들의 기여도(SHAP Value)를 모두 평균 기대 예측값에 합산하면 **최종 예측값 `{prediction:.4f}`** 이(가) 도출됩니다.\n"

    # 마크다운 파일 저장
    md_file_path = "shap_analysis_for_rag.md"
    with open(md_file_path, "w", encoding="utf-8") as f:
        f.write(md_content)

    print(f"RAG용 SHAP 분석 마크다운 저장 완료 ({md_file_path})")


if __name__ == "__main__":
    csv_file_path = './Train_0319.csv'

    if not os.path.exists(csv_file_path):
        print(f"\n[에러] '{csv_file_path}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    else:
        print("--- [1] 데이터 로드 및 전처리 시작 ---")
        train_df, val_df, test_df = load_and_preprocess_data(csv_path=csv_file_path)

        print("\n--- [2] 회귀 모델 선택 및 학습 시작 ---")
        final_model_obj, final_model_name, x_train_final, x_test_final, y_test_final = train_and_evaluate(train_df, val_df, test_df)

        print("\n--- [3] 파이프라인 모델 학습 완료 ---")

        # SHAP 분석 실행
        generate_shap_analysis(final_model_obj, x_train_final, x_test_final, y_test_final)

        print("\n모든 작업이 성공적으로 완료되었습니다. 생성된 마크다운 파일을 RAG 시스템에 로드하여 활용하세요.")

--- [1] 데이터 로드 및 전처리 시작 ---

--- [2] 회귀 모델 선택 및 학습 시작 ---
--- [1단계] 회귀 모델 선택 (기본 파라미터) ---
  RandomForest Validation R2: 0.9207
  GradientBoosting Validation R2: 0.8721
  XGBoost Validation R2: 0.9232
  LightGBM Validation R2: 0.9310
  Ridge Validation R2: 0.8138
  ElasticNet Validation R2: -0.0004


[I 2026-04-30 08:22:50,095] A new study created in memory with name: no-name-c7362b91-b4f7-432a-82fb-dc07e684ec6f


  SVR Validation R2: 0.8785
-> 선택된 최고 성능 회귀 모델: LightGBM (R2: 0.9310)

--- [2단계] LightGBM 하이퍼파라미터 튜닝 ---


[I 2026-04-30 08:22:50,793] Trial 0 finished with value: 0.9313413932889706 and parameters: {'n_estimators': 149, 'learning_rate': 0.07377786098938609, 'max_depth': 11}. Best is trial 0 with value: 0.9313413932889706.
[I 2026-04-30 08:22:51,141] Trial 1 finished with value: 0.6442703219233766 and parameters: {'n_estimators': 79, 'learning_rate': 0.011141565003001293, 'max_depth': 3}. Best is trial 0 with value: 0.9313413932889706.
[I 2026-04-30 08:22:52,185] Trial 2 finished with value: 0.9327341287582045 and parameters: {'n_estimators': 270, 'learning_rate': 0.061728257571717224, 'max_depth': 13}. Best is trial 2 with value: 0.9327341287582045.
[I 2026-04-30 08:22:53,159] Trial 3 finished with value: 0.9320543881262349 and parameters: {'n_estimators': 289, 'learning_rate': 0.14331482417539226, 'max_depth': 9}. Best is trial 2 with value: 0.9327341287582045.
[I 2026-04-30 08:22:54,308] Trial 4 finished with value: 0.9173408539689003 and parameters: {'n_estimators': 214, 'learning_rate'

최적 파라미터: {'n_estimators': 270, 'learning_rate': 0.061728257571717224, 'max_depth': 13}

=== 데이터셋별 양산 DES 속도 회귀 성능 ===
[Test Set] R2 Score: 0.9214

--- [3] 파이프라인 모델 학습 완료 ---

--- [4] SHAP 분석 시작 ---
SHAP 플롯 이미지 저장 완료 (shap_summary_bar.png, shap_summary_dot.png)

--- [5] RAG용 SHAP 분석 마크다운 생성 ---
RAG용 SHAP 분석 마크다운 저장 완료 (shap_analysis_for_rag.md)

모든 작업이 성공적으로 완료되었습니다. 생성된 마크다운 파일을 RAG 시스템에 로드하여 활용하세요.
